goal: when a car isn't moving, every sensor should read a known value. Whatever it reads instead of its offset (avg) and its noise (std)

what sensors should read at rest:
* gyro (angural rate) x,y,z: 0 (anything else is offset)
* accelerometer x,y: abt 0 if the groups is level, z: gravity 0
* throttle: 0
* brake: 0 unless the driver is holding it
* steering: not always 0 so we can't tell offset of steering from the standstill

from ou previous analysis (02_session) we found three stationary periods:
* the start 0-60
* pit stop 980-1200 (approx)
* end ~1976 to the end

which could help to compare them to find out if we have a stable offset that we can subtract

If a car is being pushed slowly, the front wheels would have a speed that is slower than walking, a car being pushed could reach the speed of 0.5m/s (assuming it's heavy and it's hard to be pushed) so a standstill would mean that the wheel speed is slower than that, say around 0.19m/s, so 1rad/s so that any real push would be over that value. 

hence, it would make sense to take the threashold as 1rad/s from the front_angular_velocity to determine if a car is staionary or not.

In [7]:
from src.loader import load
brake_press = load("pedal_brakes_pressure")
brake_press_front_max = brake_press[brake_press["front"] ==brake_press["front"].max()]
brake_press_front_max

,t_s,_timestamp,front,rear
49544,519.588272,1725799638964815,22.671854,18.742657


In [23]:
imu_acc = load("imu_acceleration")
imu_acc_x_max = imu_acc[(imu_acc["t_s"] >= 519.58) & (imu_acc["t_s"] <= 520.4)]
imu_acc_x_max

,t_s,_timestamp,x,y,z,imu_temperature
115937,519.580767,1725799638957310,-1.29,0.31,-0.94,0
115938,519.585227,1725799638961770,-1.23,0.09,-0.92,0
115939,519.589757,1725799638966300,-1.18,0.16,-0.95,0
115940,519.594281,1725799638970824,-1.22,0.10,-0.92,0
115941,519.598662,1725799638975205,-1.05,0.07,-0.92,0
...,...,...,...,...,...,...
116115,520.377963,1725799639754506,-0.79,0.23,-0.86,0
116116,520.382333,1725799639758876,-0.64,0.10,-0.84,0
116117,520.387001,1725799639763544,-0.75,0.35,-0.68,0
116118,520.391309,1725799639767852,-0.71,0.33,-0.81,0


in case of a lockup, the angular speed can show 0 during braking until the car stops or driver lifts up the brakes.
if we take that into account, and assuming the car decelerates at 1.2g (avg of (−1.29, −1.23, −1.18, −1.22, −1.05) in the table above) max at peak brake pressure, equivalent to (1.2*9.81) ~11.772m/s^2, and given our data, the max speed reached by the car is 23m/s, then the car comes to a stop after abt 2s, we we can assume that the car is at standstill if the front angular velocity ~0 for 3 seconds

In [2]:
from src.profiler import find_standstills

In [3]:
find_standstills()

,start_s,end_s,duration_s
0,0.000000,13.002358,13.002358
1,20.790024,54.929918,34.139894
2,161.421000,176.054854,14.633854
3,192.778036,206.897613,14.119577
4,227.119072,242.465951,15.346879
5,977.176868,1157.391011,180.214143
6,1163.760712,1196.196530,32.435818
7,1829.968137,1908.013226,78.045089
8,1920.559415,2206.962362,286.402947
